# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup

Import required packages

In [1]:
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv


Get an API key from [US Bureau of Labour Statistics API registration](https://data.bls.gov/registrationEngine/), and save it in the `.env` file in this folder.

The cell below reads that file and loads the key into a variable.

In [2]:
load_dotenv()

API_KEY = os.getenv("BLS_API_KEY")

## Request employment data from BLS

Call the BLS API for employment data by industry at the 4-digit NAICS code level.




Compile the list of series IDs required following this structure:

	Series ID    CEU0800000003
	Positions       Value           Field Name
	1-2             CE              Prefix
	3               U               Seasonal Adjustment Code
	4-11		08000000	Supersector and Industry Codes
	12-13           03              Data Type Code

The industry codes need to be unpacked from the [codes list](https://download.bls.gov/pub/time.series/ce/ce.industry) provided by the BLS. This is contained in a `.tsv` file but is hosted with a faulty extension, so it must be downloaded manually and read as a .tsv.

In [46]:
industry_code_map = (
    pd.read_csv("../data/reference/ce_industry.tsv", delimiter="\t")
    # Selecting display level 5 filters on 4 digit NAICS codes to match with AIOE table
    .query("display_level == 5")
    .filter(["industry_code", "naics_code", "industry_name"])
    .reset_index()
)

industry_code_map.head()

,index,industry_code,naics_code,industry_name
0,6,10113300,1133,Logging
1,10,10212100,2121,Coal mining
2,13,10212200,2122,Metal ore mining
3,16,10212300,2123,Nonmetallic mineral mining and quarrying
4,29,20236100,2361,Residential building construction


In [13]:
SERIES = "CE"
SA = "S"
industry = 10212200
data_type = "01"

series_id = f"{SERIES}{SA}{industry}{data_type}"

In [ ]:
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": [series_id],"startyear":"2019", "endyear":"2026"})

# Post query to BLS API
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)

json_data = p.json()
json_data
df = pd.json_normalize(
    json_data["Results"]["series"],
    record_path="data")
df

,year,period,periodName,latest,value,footnotes
0,2026,M06,June,true,46.3,"[{'code': 'P', 'text': 'preliminary'}]"
1,2026,M05,May,NaN,46.2,"[{'code': 'P', 'text': 'preliminary'}]"
2,2026,M04,April,NaN,46.1,[{}]
3,2026,M03,March,NaN,45.7,[{}]
4,2026,M02,February,NaN,45.7,[{}]
...,...,...,...,...,...,...
85,2019,M05,May,NaN,42.6,[{}]
86,2019,M04,April,NaN,42.7,[{}]
87,2019,M03,March,NaN,42.3,[{}]
88,2019,M02,February,NaN,42.2,[{}]
